# Fine-tune the baseline gpt model with the reward function

In [25]:
from pathlib import Path
import pickle
import copy
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import GPT2LMHeadModel
from torch.optim import AdamW
from torch.distributions import Categorical

from collections import Counter

import random
import pandas as pd

In [2]:
# paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

# device
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)
print("Device:", device)

# load baseline artifacts
with open(DATA_DIR / "artifacts.pkl", "rb") as f:
    artifacts = pickle.load(f)

item_to_token = artifacts["item_to_token"]
token_to_item = artifacts["token_to_item"]
train_sequences = artifacts["train_sequences"]

N_ITEMS = len(item_to_token)
PAD_ID = N_ITEMS
MAX_LENGTH = 32

# load pretrained baseline recommender
baseline_path = DATA_DIR / "bs_gpt2_rec"

policy = GPT2LMHeadModel.from_pretrained(baseline_path).to(device)
policy.config.pad_token_id = PAD_ID

print("Baseline vocabulary:", N_ITEMS)
print("Training sequences:", len(train_sequences))
print("Policy parameters:", sum(p.numel() for p in policy.parameters()))

# load reward model checkpoint
reward_checkpoint = torch.load(
    DATA_DIR / "reward_model1_best.pt",
    map_location=device,
    weights_only=False,
)

REWARD_PAD_ID = reward_checkpoint["reward_pad_id"]
REWARD_ITEM_TO_TOKEN = reward_checkpoint["item_to_token"]
REWARD_TOKEN_TO_ITEM = reward_checkpoint["token_to_item"]

Device: mps


Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

Baseline vocabulary: 1000
Training sequences: 7920
Policy parameters: 166208


In [3]:
class RewardModel(nn.Module):
    def __init__(self, n_items, embedding_dim=64, hidden_dim=128):
        super().__init__()

        self.item_embedding = nn.Embedding(
            n_items + 1,
            embedding_dim,
            padding_idx=n_items,
        )

        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Tanh(),
        )

    def forward(self, history, history_mask, candidate):
        embeddings = self.item_embedding(history)
        mask = history_mask.unsqueeze(-1)

        history_vector = (embeddings * mask).sum(dim=1)
        history_length = mask.sum(dim=1).clamp(min=1.0)
        history_vector = history_vector / history_length

        candidate_vector = self.item_embedding(candidate)

        combined = torch.cat(
            [history_vector, candidate_vector],
            dim=1,
        )

        return self.mlp(combined).squeeze(-1)


reward_model = RewardModel(
    n_items=reward_checkpoint["n_items"],
    embedding_dim=reward_checkpoint["embedding_dim"],
    hidden_dim=reward_checkpoint["hidden_dim"],
).to(device)

reward_model.load_state_dict(reward_checkpoint["model_state_dict"])
reward_model.eval()

for parameter in reward_model.parameters():
    parameter.requires_grad = False

print("Reward model loaded and frozen.")

Reward model loaded and frozen.


In [4]:
# check the vocab overlap

# Baseline token -> item ID -> reward-model token
baseline_to_reward = {}

for baseline_token, item_id in token_to_item.items():
    if item_id in REWARD_ITEM_TO_TOKEN:
        baseline_to_reward[int(baseline_token)] = (
            REWARD_ITEM_TO_TOKEN[item_id]
        )

print("Baseline items:", N_ITEMS)
print("Items supported by reward model:", len(baseline_to_reward))
print(
    "Vocabulary overlap:",
    f"{len(baseline_to_reward) / N_ITEMS:.1%}",
)

print("\nFirst training sequence type:", type(train_sequences[0]))
print("First training sequence sample:", str(train_sequences[0])[:500])

Baseline items: 1000
Items supported by reward model: 1000
Vocabulary overlap: 100.0%

First training sequence type: <class 'list'>
First training sequence sample: [578, 578]


In [5]:
# extratt contexts from train sequences 

contexts = [
    [int(token) for token in seq[-(MAX_LENGTH - 1):]]
    for seq in train_sequences
    if len(seq) > 0
]

print("Available contexts:", len(contexts))
print("Context length range:",
      min(map(len, contexts)), "to", max(map(len, contexts)))
print("Example context:", contexts[0])

# every context item and every possible generated item must be supported by the reward model
assert all(token in baseline_to_reward for ctx in contexts for token in ctx)
assert len(baseline_to_reward) == N_ITEMS

Available contexts: 7920
Context length range: 2 to 31
Example context: [578, 578]


In [6]:
reference_policy = copy.deepcopy(policy).to(device)
reference_policy.eval()

for parameter in reference_policy.parameters():
    parameter.requires_grad = False

optimizer = AdamW(policy.parameters(), lr=1e-5)

BATCH_SIZE = 16
RL_STEPS = 200
KL_BETA = 0.05
TEMPERATURE = 1.0

print("RL setup ready.")

RL setup ready.


In [ ]:
def make_policy_batch(batch_contexts):
    lengths = torch.tensor(
        [len(ctx) for ctx in batch_contexts],
        dtype=torch.long,
        device=device,
    )
    max_len = int(lengths.max().item())

    input_ids = torch.full(
        (len(batch_contexts), max_len),
        PAD_ID,
        dtype=torch.long,
        device=device,
    )
    attention_mask = torch.zeros_like(input_ids)

    for i, ctx in enumerate(batch_contexts):
        input_ids[i, :len(ctx)] = torch.tensor(ctx, device=device)
        attention_mask[i, :len(ctx)] = 1

    return input_ids, attention_mask, lengths


def make_reward_history(batch_contexts):
    lengths = [len(ctx) for ctx in batch_contexts]
    max_len = max(lengths)

    history = torch.full(
        (len(batch_contexts), max_len),
        REWARD_PAD_ID,
        dtype=torch.long,
        device=device,
    )
    mask = torch.zeros_like(history)

    for i, ctx in enumerate(batch_contexts):
        reward_ids = [baseline_to_reward[token] for token in ctx]
        history[i, :len(reward_ids)] = torch.tensor(
            reward_ids, dtype=torch.long, device=device
        )
        mask[i, :len(reward_ids)] = 1

    return history, mask


policy.train()
reward_history_losses = []
reward_history_scores = []
reward_history_kls = []

start_time = time.time()

for step in range(RL_STEPS):
    batch_contexts = random.choices(contexts, k=BATCH_SIZE)

    input_ids, attention_mask, lengths = make_policy_batch(batch_contexts)

    # get policy and reference next-item distributions
    outputs = policy(
        input_ids=input_ids,
        attention_mask=attention_mask,
    )
    with torch.no_grad():
        reference_outputs = reference_policy(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

    batch_indices = torch.arange(BATCH_SIZE, device=device)
    last_positions = lengths - 1

    policy_logits = outputs.logits[batch_indices, last_positions, :]
    reference_logits = reference_outputs.logits[
        batch_indices, last_positions, :
    ]

    # never recommend the padding token
    policy_logits = policy_logits.clone()
    reference_logits = reference_logits.clone()
    policy_logits[:, PAD_ID] = -1e9
    reference_logits[:, PAD_ID] = -1e9

    policy_log_probs = F.log_softmax(
        policy_logits / TEMPERATURE, dim=-1
    )
    reference_log_probs = F.log_softmax(
        reference_logits / TEMPERATURE, dim=-1
    )

    # sample one candidate per history
    distribution = Categorical(logits=policy_logits / TEMPERATURE)
    candidates = distribution.sample()
    selected_log_probs = policy_log_probs.gather(
        1, candidates.unsqueeze(1)
    ).squeeze(1)

    # score candidates using the frozen reward model
    reward_history, reward_mask = make_reward_history(batch_contexts)
    reward_candidates = torch.tensor(
        [baseline_to_reward[int(token)] for token in candidates.tolist()],
        dtype=torch.long,
        device=device,
    )

    with torch.no_grad():
        rewards = reward_model(
            reward_history,
            reward_mask,
            reward_candidates,
        )

    # center rewards within the batch to form relative advantages
    advantages = rewards - rewards.mean()
    advantages = advantages.detach()

    # exact KL divergence between the current policy and the baseline
    policy_probs = policy_log_probs.exp()
    kl = (
        policy_probs * (policy_log_probs - reference_log_probs)
    ).sum(dim=-1).mean()

    policy_gradient_loss = -(advantages * selected_log_probs).mean()
    loss = policy_gradient_loss + KL_BETA * kl

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=1.0)
    optimizer.step()

    reward_history_losses.append(float(loss.detach().cpu()))
    reward_history_scores.append(float(rewards.mean().cpu()))
    reward_history_kls.append(float(kl.detach().cpu()))

    if (step + 1) % 20 == 0:
        elapsed = time.time() - start_time
        print(
            f"Step {step + 1:>3}/{RL_STEPS} | "
            f"loss={np.mean(reward_history_losses[-20:]):.4f} | "
            f"reward={np.mean(reward_history_scores[-20:]):.4f} | "
            f"KL={np.mean(reward_history_kls[-20:]):.4f} | "
            f"elapsed={elapsed:.1f}s"
        )

print("RL fine-tuning finished.")

Step  20/200 | loss=-0.0430 | reward=0.5589 | KL=0.0286 | elapsed=11.2s
Step  40/200 | loss=-0.0691 | reward=0.5664 | KL=0.0297 | elapsed=13.5s
Step  60/200 | loss=-0.0156 | reward=0.5861 | KL=0.0299 | elapsed=16.2s
Step  80/200 | loss=-0.0475 | reward=0.5898 | KL=0.0312 | elapsed=17.2s
Step 100/200 | loss=-0.1246 | reward=0.5564 | KL=0.0314 | elapsed=18.0s
Step 120/200 | loss=-0.0158 | reward=0.5895 | KL=0.0293 | elapsed=18.9s
Step 140/200 | loss=-0.0776 | reward=0.5960 | KL=0.0304 | elapsed=19.9s
Step 160/200 | loss=-0.0142 | reward=0.5802 | KL=0.0313 | elapsed=20.8s
Step 180/200 | loss=-0.0573 | reward=0.5886 | KL=0.0343 | elapsed=21.6s
Step 200/200 | loss=-0.0628 | reward=0.5668 | KL=0.0323 | elapsed=22.5s
RL fine-tuning finished.


In [8]:
@torch.no_grad()
def recommend_greedy(model, batch_contexts):
    input_ids, attention_mask, lengths = make_policy_batch(batch_contexts)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
    )

    indices = torch.arange(len(batch_contexts), device=device)
    logits = outputs.logits[indices, lengths - 1, :].clone()
    logits[:, PAD_ID] = -1e9

    return logits.argmax(dim=-1)


@torch.no_grad()
def score_recommendations(batch_contexts, candidates):
    history, mask = make_reward_history(batch_contexts)
    reward_candidates = torch.tensor(
        [baseline_to_reward[int(token)] for token in candidates.tolist()],
        dtype=torch.long,
        device=device,
    )
    return reward_model(history, mask, reward_candidates)


policy.eval()
reference_policy.eval()

eval_contexts = random.sample(contexts, min(256, len(contexts)))

baseline_recommendations = recommend_greedy(
    reference_policy, eval_contexts
)
rl_recommendations = recommend_greedy(policy, eval_contexts)

baseline_scores = score_recommendations(
    eval_contexts, baseline_recommendations
)
rl_scores = score_recommendations(eval_contexts, rl_recommendations)

print(f"Baseline mean predicted reward: {baseline_scores.mean().item():.4f}")
print(f"RL mean predicted reward:       {rl_scores.mean().item():.4f}")
print(
    "Changed recommendations:",
    f"{(baseline_recommendations != rl_recommendations).float().mean().item():.1%}"
)

Baseline mean predicted reward: 0.6317
RL mean predicted reward:       0.6326
Changed recommendations: 2.0%


In [9]:
# save a checkpoint
rl_output_path = DATA_DIR / "bs_gpt2_rec_rl"
policy.save_pretrained(rl_output_path)

print("Saved fine-tuned policy to:", rl_output_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fine-tuned policy to: /Users/fountainer/tlab/awesomeRecs/data/bs_gpt2_rec_rl


For now we can conclude that fine-tuning with RL improved results insignificantly. Now we will proceed step by step improving the models and analysing results (and reward score VS real recommendation value).

## Establishing baseline metrics

Baseline quality analysis  will help to check if the RL model does exploit the vulnerabilities in the rewards.

In [11]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

with open(DATA_DIR / "artifacts.pkl", "rb") as f:
    artifacts = pickle.load(f)

print("=== artifacts.pkl ===")
print("Keys:", list(artifacts.keys()))

for key, value in artifacts.items():
    print(f"\n{key}: type={type(value).__name__}")
    if isinstance(value, dict):
        print("  Number of entries:", len(value))
        for k, v in list(value.items())[:2]:
            print(f"  Example: key={str(k)[:80]}, "
                  f"value_type={type(v).__name__}, "
                  f"value={str(v)[:180]}")
    elif isinstance(value, (list, tuple)):
        print("  Length:", len(value))
        if value:
            print("  First element:", str(value[0])[:250])
    else:
        print("  Value:", str(value)[:250])

print("\n=== Additional data files ===")
for path in sorted(DATA_DIR.glob("*.pkl")):
    print(f"\n{path.name} ({path.stat().st_size / 1e6:.1f} MB)")

=== artifacts.pkl ===
Keys: ['user_sequences', 'item_to_token', 'token_to_item', 'train_sequences', 't_train', 't_validate']

user_sequences: type=dict
  Number of entries: 50
  Example: key=28700, value_type=dict, value={'train': [7094195, 117174, 9143560, 121773, 1669744, 6699637, 1860416, 6690987, 7421463, 4409793, 500962, 7735065, 3987314, 3095165, 9267109, 7365232, 7451308, 8240867, 7404746, 6
  Example: key=31400, value_type=dict, value={'train': [4775947, 2045788, 7112476, 4775947, 7112476, 8905102, 2317630, 1875932, 4409684, 8630920, 6374972, 1412086, 6331256, 8213481, 3198518, 3033749, 7427984, 7317604, 5047356

item_to_token: type=dict
  Number of entries: 1000
  Example: key=5862961, value_type=int, value=0
  Example: key=9010147, value_type=int, value=1

token_to_item: type=dict
  Number of entries: 1000
  Example: key=0, value_type=int, value=5862961
  Example: key=1, value_type=int, value=9010147

train_sequences: type=list
  Length: 7920
  First element: [578, 578]

t_tr

In [12]:
interactions_path = DATA_DIR / "reward_interactions.pkl"

if interactions_path.exists():
    with open(interactions_path, "rb") as f:
        interactions = pickle.load(f)

    print("Interaction object type:", type(interactions).__name__)

    if isinstance(interactions, dict):
        print("Keys:", list(interactions.keys()))
        for key, value in interactions.items():
            print(f"\n{key}: {type(value).__name__}")
            if hasattr(value, "head"):
                print(value.head(3))
            else:
                print(str(value)[:500])
    elif hasattr(interactions, "head"):
        print(interactions.head())
        print("Columns:", list(interactions.columns))
else:
    print("reward_interactions.pkl not found.")

Interaction object type: dict
Keys: ['train', 'validation', 'test']

train: list
[{'uid': 28700, 'timestamp': 46295, 'item_id': 7094195, 'played_ratio_pct': 47}, {'uid': 28700, 'timestamp': 46300, 'item_id': 117174, 'played_ratio_pct': 2}, {'uid': 28700, 'timestamp': 46420, 'item_id': 9143560, 'played_ratio_pct': 100}, {'uid': 28700, 'timestamp': 46715, 'item_id': 121773, 'played_ratio_pct': 100}, {'uid': 28700, 'timestamp': 54645, 'item_id': 1669744, 'played_ratio_pct': 100}, {'uid': 28700, 'timestamp': 55050, 'item_id': 6699637, 'played_ratio_pct': 100}, {'uid': 28700, 'ti

validation: list
[{'uid': 28700, 'timestamp': 20619030, 'item_id': 4976274, 'played_ratio_pct': 100}, {'uid': 28700, 'timestamp': 20619325, 'item_id': 1596797, 'played_ratio_pct': 100}, {'uid': 28700, 'timestamp': 20619555, 'item_id': 4134898, 'played_ratio_pct': 100}, {'uid': 28700, 'timestamp': 20619760, 'item_id': 1291575, 'played_ratio_pct': 100}, {'uid': 28700, 'timestamp': 20619995, 'item_id': 5160105, 'play

In [13]:
# here we will build examples for metrics evaluation

baseline_model = GPT2LMHeadModel.from_pretrained(
    DATA_DIR / "bs_gpt2_rec"
).to(device)
baseline_model.eval()

rl_model = GPT2LMHeadModel.from_pretrained(
    DATA_DIR / "bs_gpt2_rec_rl"
).to(device)
rl_model.eval()

with open(DATA_DIR / "artifacts.pkl", "rb") as f:
    artifacts = pickle.load(f)

user_sequences = artifacts["user_sequences"]
item_to_token = artifacts["item_to_token"]

MAX_LENGTH = 32
N_ITEMS = len(item_to_token)
PAD_ID = N_ITEMS

def build_eval_examples(split):
    """
    Each example contains the preceding history and one future target.
    Histories are built chronologically, without including the target.
    """
    examples = []

    for uid, user_data in user_sequences.items():
        train_items = user_data["train"]
        validation_items = user_data.get("validation", [])
        test_items = user_data.get("test", [])

        history = list(train_items)

        if split == "validation":
            future_items = validation_items
        elif split == "test":
            history.extend(validation_items)
            future_items = test_items
        else:
            raise ValueError("split must be 'validation' or 'test'")

        for item_id in future_items:
            # Only evaluate targets the baseline can represent.
            if item_id in item_to_token and history:
                history_tokens = [
                    item_to_token[item]
                    for item in history
                    if item in item_to_token
                ]

                if history_tokens:
                    examples.append({
                        "uid": uid,
                        "history": history_tokens[-(MAX_LENGTH - 1):],
                        "target": item_to_token[item_id],
                        "history_items": set(history),
                    })

            # Add this event to history only after evaluating its target.
            history.append(item_id)

    return examples


validation_examples = build_eval_examples("validation")
test_examples = build_eval_examples("test")

print("Validation examples:", len(validation_examples))
print("Test examples:", len(test_examples))
print(
    "Validation target coverage:",
    f"{len(validation_examples) / max(1, sum(len(x.get('validation', [])) for x in user_sequences.values())):.1%}"
)
print(
    "Test target coverage:",
    f"{len(test_examples) / max(1, sum(len(x.get('test', [])) for x in user_sequences.values())):.1%}"
)

Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

Validation examples: 6724
Test examples: 5423
Validation target coverage: 16.1%
Test target coverage: 13.0%


In [17]:
@torch.no_grad()
def get_topk_recommendations(model, examples, k=10, batch_size=128):
    all_topk = []

    for start in range(0, len(examples), batch_size):
        batch_examples = examples[start:start + batch_size]
        lengths = [len(x["history"]) for x in batch_examples]
        max_len = max(lengths)

        input_ids = torch.full(
            (len(batch_examples), max_len),
            PAD_ID,
            dtype=torch.long,
            device=device,
        )
        attention_mask = torch.zeros_like(input_ids)

        for i, example in enumerate(batch_examples):
            seq = torch.tensor(
                example["history"], dtype=torch.long, device=device
            )
            input_ids[i, :len(seq)] = seq
            attention_mask[i, :len(seq)] = 1

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        rows = torch.arange(len(batch_examples), device=device)
        positions = torch.tensor(lengths, device=device) - 1
        logits = outputs.logits[rows, positions, :N_ITEMS]

        topk = logits.topk(k, dim=-1).indices.cpu().numpy()
        all_topk.extend(topk.tolist())

    return all_topk


K = 10

val_baseline_topk = get_topk_recommendations(
    baseline_model, validation_examples, k=K
)
val_rl_topk = get_topk_recommendations(
    rl_model, validation_examples, k=K
)

test_baseline_topk = get_topk_recommendations(
    baseline_model, test_examples, k=K
)
test_rl_topk = get_topk_recommendations(
    rl_model, test_examples, k=K
)

print("Top-K predictions computed.")

Top-K predictions computed.


In [30]:
def evaluate_rankings(examples, recommendations, k=10):
    hits = []
    ndcg = []
    repeated = []
    recommended_items = set()

    for example, ranking in zip(examples, recommendations):
        target = example["target"]
        history_tokens = set(example["history"])
        ranking = [int(t) for t in ranking[:k]]

        hits.append(float(target in ranking))

        if target in ranking:
            rank = ranking.index(target) + 1
            ndcg.append(1.0 / np.log2(rank + 1))
        else:
            ndcg.append(0.0)

        repeated.extend([
            int(token in history_tokens) for token in ranking
        ])
        recommended_items.update(ranking)

    return {
        f"Recall@{k}": float(np.mean(hits)),
        f"NDCG@{k}": float(np.mean(ndcg)),
        "Repeat rate": float(np.mean(repeated)),
        "Catalog coverage": len(recommended_items) / N_ITEMS,
        "Unique recommended items": len(recommended_items),
        "Examples": len(examples),
    }


results = {
    "Baseline / validation": evaluate_rankings(
        validation_examples, val_baseline_topk, K
    ),
    "RL / validation": evaluate_rankings(
        validation_examples, val_rl_topk, K
    ),
    "Baseline / test": evaluate_rankings(
        test_examples, test_baseline_topk, K
    ),
    "RL / test": evaluate_rankings(
        test_examples, test_rl_topk, K
    ),
}

import pandas as pd
metrics_df = pd.DataFrame(results).T
display(metrics_df.round(4))

,Recall@10,NDCG@10,Repeat rate,Catalog coverage,Unique recommended items,Examples
Baseline / validation,0.2760,0.1702,0.3091,0.821,821.0,6724.0
RL / validation,0.2775,0.1697,0.3104,0.820,820.0,6724.0
Baseline / test,0.2753,0.1754,0.2982,0.815,815.0,5423.0
RL / test,0.2748,0.1744,0.2987,0.814,814.0,5423.0


## Examine the Reward Model

In [20]:
# check current coverage
def coverage_diagnostics(split):
    total_targets = 0
    targets_in_vocab = 0
    eligible_examples = 0
    total_interactions = 0

    for uid, user_data in user_sequences.items():
        train_items = user_data["train"]
        future_items = user_data.get(split, [])

        total_interactions += len(future_items)
        targets_in_vocab += sum(
            item in item_to_token for item in future_items
        )

        history = list(train_items)
        if split == "test":
            history.extend(user_data.get("validation", []))

        for item in future_items:
            if item in item_to_token and any(
                h in item_to_token for h in history
            ):
                eligible_examples += 1
            history.append(item)

    return {
        "split": split,
        "all_interactions": total_interactions,
        "targets_in_baseline_vocab": targets_in_vocab,
        "target_vocab_coverage": targets_in_vocab / max(1, total_interactions),
        "eligible_examples": eligible_examples,
        "eligible_fraction": eligible_examples / max(1, total_interactions),
    }

coverage_df = pd.DataFrame([
    coverage_diagnostics("validation"),
    coverage_diagnostics("test"),
])

display(coverage_df.round(4))

,split,all_interactions,targets_in_baseline_vocab,target_vocab_coverage,eligible_examples,eligible_fraction
0,validation,41728,6728,0.1612,6724,0.1611
1,test,41731,5424,0.1300,5423,0.1300


In [23]:
# compare item popularity VS its reward score

# popularity in the training history, using original item IDs
train_item_counts = Counter(
    item
    for user_data in user_sequences.values()
    for item in user_data["train"]
)

# convert baseline token IDs back to original item IDs
def token_popularity(token):
    item_id = token_to_item[int(token)]
    return train_item_counts.get(item_id, 0)


@torch.no_grad()
def score_topk_candidates(examples, recommendations, batch_size=512):
    """
    Score every recommended candidate with the frozen reward model.
    Returns one predicted reward per (context, candidate) pair.
    """
    scores = []

    for start in range(0, len(examples), batch_size):
        batch_examples = examples[start:start + batch_size]
        batch_recs = recommendations[start:start + batch_size]

        histories = []
        masks = []
        candidates = []

        max_len = max(len(ex["history"]) for ex in batch_examples)

        history_tensor = torch.full(
            (len(batch_examples) * len(batch_recs[0]), max_len),
            REWARD_PAD_ID,
            dtype=torch.long,
            device=device,
        )
        mask_tensor = torch.zeros_like(history_tensor)

        for i, (example, recs) in enumerate(zip(batch_examples, batch_recs)):
            reward_history = [
                baseline_to_reward[token] for token in example["history"]
            ]

            for j, token in enumerate(recs):
                row = i * len(recs) + j
                history_tensor[row, :len(reward_history)] = torch.tensor(
                    reward_history, dtype=torch.long, device=device
                )
                mask_tensor[row, :len(reward_history)] = 1
                candidates.append(baseline_to_reward[int(token)])

        candidate_tensor = torch.tensor(
            candidates, dtype=torch.long, device=device
        )

        batch_scores = reward_model(
            history_tensor,
            mask_tensor,
            candidate_tensor,
        )
        scores.extend(batch_scores.cpu().tolist())

    return np.asarray(scores).reshape(len(examples), -1)


baseline_val_reward_scores = score_topk_candidates(
    validation_examples, val_baseline_topk
)
rl_val_reward_scores = score_topk_candidates(
    validation_examples, val_rl_topk
)


def recommendation_audit(examples, recommendations, reward_scores):
    flat_recs = [int(t) for row in recommendations for t in row]
    flat_popularity = [token_popularity(t) for t in flat_recs]

    # calculate overlap with the history for each recommended item
    repeat_flags = [
        int(int(token) in example["history"])
        for example, row in zip(examples, recommendations)
        for token in row
    ]

    return {
        "Mean predicted reward": float(reward_scores.mean()),
        "Median predicted reward": float(np.median(reward_scores)),
        "Mean training popularity": float(np.mean(flat_popularity)),
        "Median training popularity": float(np.median(flat_popularity)),
        "Fraction with zero training occurrences": float(
            np.mean(np.asarray(flat_popularity) == 0)
        ),
        "Repeat rate": float(np.mean(repeat_flags)),
        "Unique recommended items": len(set(flat_recs)),
        "Catalog coverage": len(set(flat_recs)) / N_ITEMS,
    }


audit_df = pd.DataFrame({
    "Baseline": recommendation_audit(
        validation_examples, val_baseline_topk, baseline_val_reward_scores
    ),
    "RL": recommendation_audit(
        validation_examples, val_rl_topk, rl_val_reward_scores
    ),
}).T

display(audit_df.round(4))

,Mean predicted reward,Median predicted reward,Mean training popularity,Median training popularity,Fraction with zero training occurrences,Repeat rate,Unique recommended items,Catalog coverage
Baseline,0.5487,0.5845,95.1613,76.0,0.0,0.3091,821.0,0.821
RL,0.5509,0.5865,96.7485,77.0,0.0,0.3104,820.0,0.820


In [ ]:
# check which items are actually getting high rewards

def top_rewarded_items(recommendations, reward_scores, n=15):
    item_scores = {}

    for row, scores in zip(recommendations, reward_scores):
        for token, score in zip(row, scores):
            item_scores.setdefault(int(token), []).append(float(score))

    rows = []
    for token, scores in item_scores.items():
        item_id = token_to_item[token]
        rows.append({
            "token": token,
            "item_id": item_id,
            "mean_predicted_reward": np.mean(scores),
            "recommendation_count": len(scores),
            "training_interactions": train_item_counts.get(item_id, 0),
        })

    return pd.DataFrame(rows).sort_values(
        "mean_predicted_reward", ascending=False
    ).head(n).reset_index(drop=True)


print("Top-rewarded items among baseline recommendations:")
display(top_rewarded_items(val_baseline_topk, baseline_val_reward_scores))

print("Top-rewarded items among RL recommendations:")
display(top_rewarded_items(val_rl_topk, rl_val_reward_scores))

Top-rewarded items among baseline recommendations:


,token,item_id,mean_predicted_reward,recommendation_count,training_interactions
0,664,619494,0.974389,1,39
1,230,4211127,0.924082,22,69
2,347,1958145,0.900601,11,57
3,184,4334417,0.898944,70,76
4,179,555291,0.896413,73,77
5,150,1073534,0.889654,72,81
6,982,5603252,0.888960,12,32
7,235,4017613,0.887828,53,68
8,657,9180107,0.885566,14,39
9,183,4380055,0.883287,69,76


Top-rewarded items among RL recommendations:


,token,item_id,mean_predicted_reward,recommendation_count,training_interactions
0,664,619494,0.974389,1,39
1,230,4211127,0.924847,23,69
2,657,9180107,0.906498,12,39
3,347,1958145,0.900601,11,57
4,179,555291,0.896330,75,77
5,184,4334417,0.891006,78,76
6,150,1073534,0.889654,72,81
7,183,4380055,0.889320,65,76
8,982,5603252,0.889045,11,32
9,235,4017613,0.887683,52,68


A small summary:
- RL model results are very similar to the baseline.
- the repeat rate is about 0.31
- the link between popularity and recommendation count is not clearly seen

## Trying out different RL model modifications

In [26]:


def train_rl_variant(
    name,
    kl_beta=0.05,
    repeat_penalty=0.0,
    steps=300,
    lr=1e-5,
    seed=42,
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = GPT2LMHeadModel.from_pretrained(
        DATA_DIR / "bs_gpt2_rec"
    ).to(device)
    model.train()

    reference = GPT2LMHeadModel.from_pretrained(
        DATA_DIR / "bs_gpt2_rec"
    ).to(device)
    reference.eval()
    for p in reference.parameters():
        p.requires_grad = False

    optimizer = AdamW(model.parameters(), lr=lr)
    logs = []

    for step in range(steps):
        batch_contexts = random.choices(contexts, k=BATCH_SIZE)
        input_ids, attention_mask, lengths = make_policy_batch(batch_contexts)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        with torch.no_grad():
            ref_outputs = reference(
                input_ids=input_ids, attention_mask=attention_mask
            )

        rows = torch.arange(len(batch_contexts), device=device)
        positions = lengths - 1
        logits = outputs.logits[rows, positions, :].clone()
        ref_logits = ref_outputs.logits[rows, positions, :].clone()

        logits[:, PAD_ID] = -1e9
        ref_logits[:, PAD_ID] = -1e9

        log_probs = F.log_softmax(logits, dim=-1)
        ref_log_probs = F.log_softmax(ref_logits, dim=-1)

        distribution = Categorical(logits=logits)
        candidates = distribution.sample()
        selected_log_probs = log_probs.gather(
            1, candidates[:, None]
        ).squeeze(1)

        reward_history, reward_mask = make_reward_history(batch_contexts)
        reward_candidates = torch.tensor(
            [baseline_to_reward[int(t)] for t in candidates.tolist()],
            dtype=torch.long,
            device=device,
        )

        with torch.no_grad():
            rewards = reward_model(
                reward_history, reward_mask, reward_candidates
            )

            if repeat_penalty > 0:
                repeated = torch.tensor(
                    [
                        int(int(t) in set(ctx))
                        for t, ctx in zip(candidates.tolist(), batch_contexts)
                    ],
                    dtype=torch.float32,
                    device=device,
                )
                rewards = rewards - repeat_penalty * repeated

        advantages = (rewards - rewards.mean()).detach()

        probs = log_probs.exp()
        kl = (probs * (log_probs - ref_log_probs)).sum(dim=-1).mean()

        policy_loss = -(advantages * selected_log_probs).mean()
        loss = policy_loss + kl_beta * kl

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        if (step + 1) % 25 == 0:
            logs.append({
                "step": step + 1,
                "reward": rewards.mean().item(),
                "kl": kl.item(),
                "loss": loss.item(),
            })

    model.eval()
    save_path = DATA_DIR / f"bs_gpt2_rec_{name}"
    model.save_pretrained(save_path)

    return model, pd.DataFrame(logs), save_path

In [27]:
variants = {
    "rl_control": {
        "kl_beta": 0.05,
        "repeat_penalty": 0.0,
    },
    "rl_low_kl": {
        "kl_beta": 0.005,
        "repeat_penalty": 0.0,
    },
    "rl_low_kl_repeat_penalty": {
        "kl_beta": 0.005,
        "repeat_penalty": 0.2,
    },
}

trained_variants = {}
training_logs = {}

for name, params in variants.items():
    print(f"\nTraining {name} ...")
    model, logs, path = train_rl_variant(
        name=name,
        **params,
        steps=300,
        lr=1e-5,
        seed=42,
    )
    trained_variants[name] = model
    training_logs[name] = logs
    print("Saved:", path)
    display(logs.tail(3))


Training rl_control ...


Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: /Users/fountainer/tlab/awesomeRecs/data/bs_gpt2_rec_rl_control


...

Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: /Users/fountainer/tlab/awesomeRecs/data/bs_gpt2_rec_rl_low_kl_repeat_penalty


,step,reward,kl,loss
9,250,0.428450,0.050938,0.008684
10,275,0.467562,0.033673,0.007588
11,300,0.566415,0.034784,0.259024


In [28]:
audit_results = []

for name, model in {
    "Baseline": baseline_model,
    **trained_variants,
}.items():
    recs = get_topk_recommendations(
        model, validation_examples, k=10
    )
    scores = score_topk_candidates(validation_examples, recs)

    audit_results.append({
        "Model": name,
        **recommendation_audit(validation_examples, recs, scores),
    })

audit_results_df = pd.DataFrame(audit_results).set_index("Model")
display(audit_results_df.round(4))

,Mean predicted reward,Median predicted reward,Mean training popularity,Median training popularity,Fraction with zero training occurrences,Repeat rate,Unique recommended items,Catalog coverage
Model,,,,,,,,
Baseline,0.5487,0.5845,95.1613,76.0,0.0,0.3091,821,0.821
rl_control,0.5505,0.5864,95.6791,77.0,0.0,0.3087,824,0.824
rl_low_kl,0.5505,0.5865,95.6869,77.0,0.0,0.3087,824,0.824
rl_low_kl_repeat_penalty,0.5502,0.5863,95.3277,77.0,0.0,0.3086,822,0.822


In [31]:
models_to_evaluate = {
    "Baseline": baseline_model,
    **trained_variants,
}

validation_rows = []

for name, model in models_to_evaluate.items():
    recs = get_topk_recommendations(
        model, validation_examples, k=10
    )
    metrics = evaluate_rankings(
        validation_examples, recs, k=10
    )
    validation_rows.append({"Model": name, **metrics})

validation_metrics = pd.DataFrame(validation_rows).set_index("Model")
display(validation_metrics.round(4))

,Recall@10,NDCG@10,Repeat rate,Catalog coverage,Unique recommended items,Examples
Model,,,,,,
Baseline,0.2760,0.1702,0.3091,0.821,821,6724
rl_control,0.2751,0.1685,0.3087,0.824,824,6724
rl_low_kl,0.2748,0.1684,0.3087,0.824,824,6724
rl_low_kl_repeat_penalty,0.2739,0.1682,0.3086,0.822,822,6724


The repeat penalty did not improve accuracy and did not meaningfully change catalog coverage.